# 사용자 그룹 간 행동 차이의 통계 검증

기존 분석에서 중앙값으로 비교한 행동 차이를 통계 검정으로 보완함. 지표 기준과 그룹 분류는 변경하지 않음.

**실행 완료: 기존에 제공한 원본 데이터와 최종 v4 노트북 기준으로 사용자별 지표를 복원해 검정을 수행함.** 사용자 수, 최종 그룹별 인원과 기존 보고서의 핵심 중앙값을 재현한 뒤 실행함. 과거 4개 그룹 기준의 user_group.csv는 사용하지 않음.

## 1. 분석 범위

- 분석 단위: 사용자 1명당 1행.
- 전체 그룹: G0, G1, G2. 각 그룹의 세 쌍을 비교함.
- G2 하위 유형: A, B, C. 각 유형의 세 쌍을 비교함. 기타 복합형 48명은 하위 유형 비교에서 제외하지만 G2 전체 비교에는 포함함.
- 검정 지표: 상세조회 요청 수, 지원 1건당 상세조회, 로그 활동일수.
- 로그 미관측 사용자는 로그 기반 지표 검정에서 제외함. 결측치를 0으로 대체하지 않음.

지원 횟수, 집중성, 직무 다양성, 조회 후 지원 간격은 분류 기준과 직접 겹쳐 이번 검정 대상에서 제외함. 선정한 지표도 그룹 구성과 완전히 독립적이지 않음. 특히 지원당 조회는 지원량을 분모로 사용하고, 조회 수는 북마크 탐색성 조건과 관련되므로 검정 결과를 분류의 타당성이나 허수지원의 증거로 해석하지 않음.

## 2. 검정 방법

| 방법 | 목적 |
| --- | --- |
| Kruskal-Wallis | G0/G1/G2 및 A/B/C 각각에서 그룹 간 순위 기반 차이를 전체적으로 확인함. |
| Mann-Whitney U | 어느 두 그룹 사이에 순위 기반 차이가 나타나는지 확인하는 주 검정임. 분포 모양이 다르면 중앙값만의 차이로 해석하지 않음. |
| Welch t검정 | 등분산을 가정하지 않고 원자료의 평균 차이를 확인하는 보조 검정임. |
| Holm 보정 | 다중 검정에 따른 오류를 줄이기 위해 검정 방법별로 보정함. |

유의수준은 0.05이며 양측 검정을 사용함. Kruskal-Wallis 6개, Mann-Whitney U 18개, Welch t검정 18개를 각각 하나의 검정군으로 보정함. 쌍별 비교는 전체 검정의 유의 여부와 무관하게 사전에 정한 모든 쌍을 실행함. 주 결론은 Mann-Whitney 결과를 기준으로 작성하고 Welch 결과는 평균에 대한 보조 결과로 구분함.

Mann-Whitney 효과 크기는 `2 × U / (n1 × n2) - 1`로 계산함. 양수면 그룹1이 상대적으로 큰 값을 보이며, 0에 가까울수록 순위 차이가 작음. Welch 평균 차이의 95% 신뢰구간은 개별 비교의 구간으로 다중 비교 보정 구간이 아님.


In [1]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


## 3. 데이터 불러오기

기존 `code_master_v4_통일수정본.ipynb`의 사용자별 지표 생성 셀에서 저장하는 `data/prep/report_user_features_v4.csv`를 사용함. 기존 노트북과 같은 커널에서 실행할 때는 메모리에 있는 `report_user_features`를 사용함. 별도로 실행할 때는 아래 파일 경로를 확인함.

필요 패키지: Python, pandas, NumPy, SciPy. 원본 사용자별 CSV는 로컬 입력용이며 GitHub에는 검정 코드와 집계 결과를 올리면 됨.


In [2]:
DATA_FILE = Path('data/prep/report_user_features_v4.csv')

if 'report_user_features' in globals():
    df = report_user_features.copy()
else:
    if not DATA_FILE.exists():
        raise FileNotFoundError(
            'report_user_features_v4.csv가 필요합니다. '
            '기존 노트북의 사용자별 지표 생성 셀을 실행하거나 DATA_FILE 경로를 수정하세요.'
        )
    df = pd.read_csv(DATA_FILE, dtype={'user_uuid': str})

required = [
    'user_uuid', 'analysis_group', 'final_segment', 'log_observed',
    'detail_get_requests', 'detail_per_app', 'log_active_days'
]
missing = sorted(set(required) - set(df.columns))
if missing:
    raise ValueError(f'필요한 열이 없습니다: {missing}')
assert df['user_uuid'].notna().all(), '사용자 ID 결측을 확인하세요.'
assert df['user_uuid'].is_unique, '사용자별 한 행으로 통합해야 합니다.'
assert len(df) == 10287, '최종 분석 대상 10,287명과 다른 데이터입니다.'

group_map = {'G0_신호없음': 'G0', 'G1_단일신호': 'G1', 'G2_복합신호': 'G2'}
segment_map = {
    'G0_신호없음': 'G0', 'G1_단일신호': 'G1',
    'A_지원행동복합형': 'A', 'B_공고탐색결합형': 'B',
    'C_고복합행동형': 'C', '기타_복합형': '기타'
}
df['group'] = df['analysis_group'].map(group_map)
df['segment'] = df['final_segment'].map(segment_map)
assert df['group'].notna().all(), '그룹명을 확인하세요.'
assert df['segment'].notna().all(), '세부 유형명을 확인하세요.'
assert df['group'].value_counts().to_dict() == {'G0': 6484, 'G1': 2941, 'G2': 862}
assert df['segment'].value_counts().to_dict() == {
    'G0': 6484, 'G1': 2941, 'A': 151, 'B': 492, 'C': 171, '기타': 48
}
for group in ['G0', 'G1']:
    assert df.loc[df['group'].eq(group), 'segment'].eq(group).all()
assert df.loc[df['group'].eq('G2'), 'segment'].isin(['A', 'B', 'C', '기타']).all()

observed = df['log_observed'].astype(str).str.lower().map({
    'true': True, 'false': False, '1': True, '0': False,
    '1.0': True, '0.0': False
})
assert observed.notna().all(), '로그 관측 여부를 확인하세요.'
print('전체 사용자 수:', len(df))
print('로그 미관측으로 제외한 사용자 수:', int((~observed).sum()))
df = df.loc[observed].copy()

metrics = {
    'detail_get_requests': '상세조회 요청 수',
    'detail_per_app': '지원 1건당 상세조회',
    'log_active_days': '로그 활동일수'
}
for metric in metrics:
    df[metric] = pd.to_numeric(df[metric], errors='raise')
    df[metric] = df[metric].replace([np.inf, -np.inf], np.nan)
    assert df[metric].dropna().ge(0).all(), f'{metric}에 음수가 있습니다.'
assert df['log_active_days'].dropna().gt(0).all()

families = [
    ('전체 그룹', 'group', ['G0', 'G1', 'G2']),
    ('G2 하위 유형', 'segment', ['A', 'B', 'C'])
]


전체 사용자 수: 10287
로그 미관측으로 제외한 사용자 수: 1


## 4. 그룹별 표본 수와 분포 확인

유효 표본 수, 결측 수, 평균, 표준편차, 중앙값과 사분위수를 함께 확인함. 검정은 집계값이 아닌 사용자별 원자료로 수행함.


In [3]:
summary_rows = []
for family, column, groups in families:
    for group in groups:
        users = df.loc[df[column].eq(group)]
        for metric, label in metrics.items():
            values = users[metric].dropna()
            if len(values) < 5:
                raise ValueError(f'{family} / {group} / {label}: 유효 표본이 5명 미만입니다.')
            summary_rows.append({
                '비교범위': family, '그룹': group, '지표': label,
                '유효표본수': len(values), '결측수': int(users[metric].isna().sum()),
                '평균': values.mean(), '표준편차': values.std(),
                'Q1': values.quantile(0.25), '중앙값': values.median(),
                'Q3': values.quantile(0.75), '최댓값': values.max()
            })
summary = pd.DataFrame(summary_rows)
display(summary.round(3))


비교범위,그룹,지표,유효표본수,결측수,평균,표준편차,Q1,중앙값,Q3,최댓값
전체 그룹,G0,상세조회 요청 수,6484,0,82.614,153.922,11,34,92,3448
전체 그룹,G0,지원 1건당 상세조회,6484,0,27.346,65.786,5,12,27.667,2813
전체 그룹,G0,로그 활동일수,6484,0,36.993,47.621,10,22,45,552
전체 그룹,G1,상세조회 요청 수,2941,0,101.119,168.023,19,47,115,3360
전체 그룹,G1,지원 1건당 상세조회,2941,0,20.285,49.171,4.333,9,19,1192
전체 그룹,G1,로그 활동일수,2941,0,39.764,47.235,12,25,48,529
전체 그룹,G2,상세조회 요청 수,861,0,193.936,290.743,41,95,216,2675
전체 그룹,G2,지원 1건당 상세조회,861,0,9.423,27.1,2.625,4.588,8.403,576
전체 그룹,G2,로그 활동일수,861,0,54.289,61.891,16,34,69,603
G2 하위 유형,A,상세조회 요청 수,150,0,214.647,248.047,61,115,280.75,1247


## 5. 전체 차이 확인

각 비교 범위에서 세 그룹을 함께 비교함. Holm 보정은 p값을 작은 순서대로 정렬한 뒤 남은 검정 개수를 곱하고, 누적 최댓값을 취하는 방식으로 계산함.


In [4]:
def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan)
    valid = np.flatnonzero(np.isfinite(p_values))
    order = valid[np.argsort(p_values[valid])]
    multiplied = p_values[order] * np.arange(len(order), 0, -1)
    adjusted[order] = np.minimum(np.maximum.accumulate(multiplied), 1.0)
    return adjusted

overall_rows = []
for family, column, groups in families:
    for metric, label in metrics.items():
        samples = []
        for group in groups:
            samples.append(df.loc[df[column].eq(group), metric].dropna().to_numpy())
        if np.unique(np.concatenate(samples)).size == 1:
            h_value, p_value = 0.0, 1.0
        else:
            h_value, p_value = stats.kruskal(*samples)
        overall_rows.append({
            '비교범위': family, '지표': label,
            'H통계량': h_value, 'p값': p_value
        })
overall = pd.DataFrame(overall_rows)
overall['Holm보정_p값'] = holm_adjust(overall['p값'])
overall['보정후_유의'] = overall['Holm보정_p값'].lt(0.05)
display(overall)


비교범위,지표,H통계량,p값,Holm보정_p값,보정후_유의
전체 그룹,상세조회 요청 수,450.159,1.77528e-98,8.87639e-98,True
전체 그룹,지원 1건당 상세조회,489.196,5.92021e-107,3.55213e-106,True
전체 그룹,로그 활동일수,137.648,1.28867e-30,5.15466e-30,True
G2 하위 유형,상세조회 요청 수,57.8666,2.71906e-13,5.43811e-13,True
G2 하위 유형,지원 1건당 상세조회,81.9884,1.57197e-18,4.71592e-18,True
G2 하위 유형,로그 활동일수,11.8668,0.00264947,0.00264947,True


## 6. 쌍별 비교와 효과 크기

모든 비교는 서로 다른 사용자 그룹 사이에서 수행함. 효과 크기와 평균 차이의 방향은 `그룹1 - 그룹2` 기준임. Welch t검정의 평균 차이와 Mann-Whitney의 순위 차이는 서로 다른 질문이므로, 결과가 다를 때 유리한 p값만 선택하지 않음.


In [5]:
pair_rows = []
for family, column, groups in families:
    for metric, label in metrics.items():
        for group1, group2 in combinations(groups, 2):
            x = df.loc[df[column].eq(group1), metric].dropna().to_numpy()
            y = df.loc[df[column].eq(group2), metric].dropna().to_numpy()
            u_value, p_u = stats.mannwhitneyu(
                x, y, alternative='two-sided', method='asymptotic'
            )
            rank_effect = 2 * u_value / (len(x) * len(y)) - 1
            if x.var(ddof=1) == 0 and y.var(ddof=1) == 0:
                t_value, p_t, ci_low, ci_high = np.nan, np.nan, np.nan, np.nan
            else:
                welch = stats.ttest_ind(x, y, equal_var=False, alternative='two-sided')
                t_value, p_t = welch.statistic, welch.pvalue
                ci = welch.confidence_interval(confidence_level=0.95)
                ci_low, ci_high = ci.low, ci.high
            pair_rows.append({
                '비교범위': family, '지표': label, '그룹1': group1, '그룹2': group2,
                'n1': len(x), 'n2': len(y),
                '중앙값1': np.median(x), '중앙값2': np.median(y),
                '평균1': x.mean(), '평균2': y.mean(),
                '평균차이_1-2': x.mean() - y.mean(),
                '평균차이_95CI하한': ci_low, '평균차이_95CI상한': ci_high,
                'U통계량': u_value, 'MW_p값': p_u, '순위효과크기_1-2': rank_effect,
                'Welch_t통계량': t_value, 'Welch_p값': p_t
            })
pairs = pd.DataFrame(pair_rows)
pairs['MW_Holm보정_p값'] = holm_adjust(pairs['MW_p값'])
pairs['MW_보정후_유의'] = pairs['MW_Holm보정_p값'].lt(0.05)
pairs['Welch_Holm보정_p값'] = holm_adjust(pairs['Welch_p값'])
pairs['Welch_보정후_유의'] = pairs['Welch_Holm보정_p값'].lt(0.05).astype('boolean')
pairs.loc[pairs['Welch_p값'].isna(), 'Welch_보정후_유의'] = pd.NA
display(pairs[[
    '비교범위', '지표', '그룹1', '그룹2', 'n1', 'n2',
    'MW_Holm보정_p값', '순위효과크기_1-2', 'MW_보정후_유의'
]])
display(pairs[[
    '비교범위', '지표', '그룹1', '그룹2', '평균차이_1-2',
    '평균차이_95CI하한', '평균차이_95CI상한', 'Welch_Holm보정_p값'
]])


비교범위,지표,그룹1,그룹2,n1,n2,MW_Holm보정_p값,순위효과크기_1-2,MW_보정후_유의
전체 그룹,상세조회 요청 수,G0,G1,6484,2941,2.61292e-26,-0.139253,True
전체 그룹,상세조회 요청 수,G0,G2,6484,861,8.21153e-85,-0.411694,True
전체 그룹,상세조회 요청 수,G1,G2,2941,861,5.97975e-40,-0.300501,True
전체 그룹,지원 1건당 상세조회,G0,G1,6484,2941,1.39607e-19,0.119626,True
전체 그룹,지원 1건당 상세조회,G0,G2,6484,861,1.17978e-96,0.439776,True
전체 그룹,지원 1건당 상세조회,G1,G2,2941,861,7.21783e-57,0.359434,True
전체 그룹,로그 활동일수,G0,G1,6484,2941,4.63475e-08,-0.0744415,True
전체 그룹,로그 활동일수,G0,G2,6484,861,4.97783e-27,-0.230476,True
전체 그룹,로그 활동일수,G1,G2,2941,861,1.27898e-12,-0.166021,True
G2 하위 유형,상세조회 요청 수,A,B,150,492,8.10888e-07,0.284106,True


비교범위,지표,그룹1,그룹2,평균차이_1-2,평균차이_95CI하한,평균차이_95CI상한,Welch_Holm보정_p값
전체 그룹,상세조회 요청 수,G0,G1,-18.5044,-25.6413,-11.3675,3.46075e-06
전체 그룹,상세조회 요청 수,G0,G2,-111.322,-131.126,-91.5176,1.93396e-25
전체 그룹,상세조회 요청 수,G1,G2,-92.8175,-113.189,-72.4461,2.7962e-17
전체 그룹,지원 1건당 상세조회,G0,G1,7.06023,4.66775,9.45271,7.55388e-08
전체 그룹,지원 1건당 상세조회,G0,G2,17.9227,15.5048,20.3405,8.90667e-45
전체 그룹,지원 1건당 상세조회,G1,G2,10.8624,8.3246,13.4003,1.14993e-15
전체 그룹,로그 활동일수,G0,G1,-2.77131,-4.83517,-0.707437,0.0425131
전체 그룹,로그 활동일수,G0,G2,-17.2961,-21.5948,-12.9975,1.05938e-13
전체 그룹,로그 활동일수,G1,G2,-14.5248,-19.0021,-10.0475,3.36062e-09
G2 하위 유형,상세조회 요청 수,A,B,79.9028,36.2376,123.568,0.00270523


## 7. 결과 저장 및 해석

다음 셀은 **실제 입력 데이터로 실행된 경우에만** 집계 결과 파일을 생성함. 사용자 ID는 저장하지 않음. 작은 p값이 화면에서 0으로 반올림될 수 있으므로, README에 적을 때는 과학적 표기나 `p < 0.001`과 같은 표기를 사용함.


In [6]:
OUTPUT_DIR = Path('results/statistical_validation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_DIR / 'group_summary.csv', index=False, encoding='utf-8-sig')
overall.to_csv(OUTPUT_DIR / 'kruskal_results.csv', index=False, encoding='utf-8-sig')
pairs.to_csv(OUTPUT_DIR / 'pairwise_results.csv', index=False, encoding='utf-8-sig')
print('집계 결과 저장 위치:', OUTPUT_DIR.resolve())
print('Mann-Whitney 보정 후 유의한 비교:', int(pairs['MW_보정후_유의'].sum()), '/', len(pairs))


집계 결과 저장 위치: /workspace/scratch/1511bc566174/statistical-validation/results/statistical_validation
Mann-Whitney 보정 후 유의한 비교: 14 / 18


### README에 결과를 작성할 때

1. 실제 실행 후 비교 그룹, 지표, 유효 표본 수, Holm 보정 p값과 순위 효과 크기를 함께 제시함.
2. Mann-Whitney 유의 결과는 순위 기반 차이로 설명함. 분포 모양을 확인하지 않은 상태에서 중앙값 차이가 검증됐다고 단정하지 않음.
3. Welch 결과는 평균 차이와 개별 95% 신뢰구간을 별도로 제시함. 보정 p값과 달리 신뢰구간은 다중 비교 보정 구간이 아님.
4. 유의하지 않은 결과는 차이가 없다고 확정하는 대신, 이번 자료와 검정에서 차이의 근거를 확보하지 못했다고 설명함.
5. 기존 자료를 보고 비교를 정한 후속 탐색 분석이며 독립 표본의 확인 검증이 아님. 사용자별 관측의 독립성을 가정하며, 활동기간과 직무 등 다른 요인을 통제한 인과 분석도 아님.
6. p값이 작아도 효과 크기가 작을 수 있음. 통계적 차이를 서비스 개선 효과나 그룹 분류의 타당성과 동일시하지 않음.

### 참고 문서

- [SciPy: Kruskal-Wallis](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kruskal.html)
- [SciPy: Mann-Whitney U](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mannwhitneyu.html)
- [SciPy: Welch t검정](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html)
- [statsmodels: 다중 검정 보정 방법](https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html)


## 실제 실행 결과 요약

- 로그 미관측 사용자 1명은 A 유형에 속함. 따라서 검정 표본은 A 150명, G2 861명이며 나머지 유형의 인원은 변하지 않음.
- 원래 보고서는 상세조회 미관측 값을 0으로 포함한 기술통계였지만, 이번 검정에서는 해당 사용자를 제외함. 이에 A의 지원당 조회 중앙값이 기존 3.29에서 검정 표본 기준 약 3.30으로 달라짐.
- G0와 G2는 검정한 세 지표 모두 Holm 보정 후 차이가 확인됨.
- A와 C는 세 지표 모두 차이가 확인되지 않았고, B와 C의 활동일수도 차이가 확인되지 않음. 유의하지 않은 결과를 두 집단이 같다는 결론으로 바꾸지 않음.

전체 표본 수와 원래 보고서 중앙값 재현은 사용자 제외 전에 확인했으며, 검정은 로그 관측 사용자 10,286명으로 수행함. 상세 해석은 `statistical_validation_results.md`를 참고함.
